# PriceCatcher Price Scraper — Update ingredients.json

**Source:** [data.gov.my PriceCatcher](https://data.gov.my/data-catalogue/pricecatcher)

Daily retail prices from thousands of premises across all 16 states — real transactional data, April 2026.

**Run all cells top to bottom. Cell 5 writes to ingredients.json — review Cell 4 first.**


In [3]:
# Cell 1 — Install dependencies
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'requests', 'pandas', '-q'])
print('Ready')

Ready


In [4]:
# Cell 2 — Download live data from storage.data.gov.my
import requests
import pandas as pd
import io
from datetime import date

YYYYMM = date.today().strftime('%Y-%m')  # e.g. 2026-04

BASE = 'https://storage.data.gov.my/pricecatcher'
HEADERS = {'User-Agent': 'Mozilla/5.0 (research / data analysis)'}

print(f'Downloading PriceCatcher data for {YYYYMM}...')

def fetch_csv(url):
    r = requests.get(url, headers=HEADERS, timeout=30)
    r.raise_for_status()
    return pd.read_csv(io.StringIO(r.text))

# Main price data
df_price = fetch_csv(f'{BASE}/pricecatcher_{YYYYMM}.csv')
print(f'  prices: {len(df_price):,} rows')

# Item lookup: item_code → item name + unit + category
df_item = fetch_csv(f'{BASE}/lookup_item.csv')
df_item = df_item[df_item['item_code'] > 0]
print(f'  items: {len(df_item)} items')

# Premise lookup: premise_code → state
df_premise = fetch_csv(f'{BASE}/lookup_premise.csv')
df_premise = df_premise[df_premise['premise_code'] > 0].copy()
df_premise['premise_code'] = df_premise['premise_code'].astype(int)
print(f'  premises: {len(df_premise)} premises across {df_premise["state"].nunique()} states')

print('\nStates in dataset:', sorted(df_premise['state'].dropna().unique().tolist()))

  prices: 1,297,522 rows
  items: 756 items
  premises: 3858 premises across 16 states

States in dataset: ['Johor', 'Kedah', 'Kelantan', 'Melaka', 'Negeri Sembilan', 'Pahang', 'Perak', 'Perlis', 'Pulau Pinang', 'Sabah', 'Sarawak', 'Selangor', 'Terengganu', 'W.P. Kuala Lumpur', 'W.P. Labuan', 'W.P. Putrajaya']


In [5]:
# Cell 3 — Define item code → ingredients.json ID mapping
#
# Maps our ingredient IDs to the SPECIFIC PriceCatcher item codes
# and the unit size so we can normalise everything to price-per-kg

ITEM_MAP = {
    # our_id: {codes: [list of item_codes], unit_kg: size of 1 unit in kg}
    'ayam_standard':  {'codes': [1],                           'unit_kg': 1.0},
    'daging_lembu':   {'codes': [1370, 1371, 1372, 1373, 1913],'unit_kg': 1.0},
    'santan_segar':   {'codes': [103],                         'unit_kg': 1.0},
    'bawang_merah':   {'codes': [131],                         'unit_kg': 1.0},
    'bawang_putih':   {'codes': [1564],                        'unit_kg': 1.0},
    'cili_kering':    {'codes': [1899, 1900],                  'unit_kg': 1.0},
    'lengkuas':       {'codes': [1819],                        'unit_kg': 1.0},
    'kunyit_hidup':   {'codes': [108],                         'unit_kg': 1.0},
    'halia':          {'codes': [95],                          'unit_kg': 1.0},   # halia basah tua
    'beras_wangi':    {'codes': [992, 1445, 1581, 1582],       'unit_kg': 10.0},  # 10kg packs → per kg
    'beras_basmati':  {'codes': [1474],                        'unit_kg': 5.0},   # 5kg pack → per kg
    'minyak_masak':   {'codes': [918, 1091, 1094, 1937, 1940],'unit_kg': 1.0},
    'gula_pasir':     {'codes': [1589, 1590],                  'unit_kg': 1.0},
    'telur_ayam':     {'codes': [1109, 1110],                  'unit_kg': 1.0},   # per 30 biji
    'ikan_kembung':   {'codes': [55],                          'unit_kg': 1.0},
}

STATE_NORMALISE = {
    'W.P. Kuala Lumpur': 'Kuala Lumpur',
    'W.P. Putrajaya': 'Putrajaya',
    'W.P. Labuan': 'Labuan',
    'Pulau Pinang': 'Pulau Pinang',
}

print(f'Mapping {len(ITEM_MAP)} ingredients to PriceCatcher codes')

Mapping 15 ingredients to PriceCatcher codes


In [6]:
# Cell 4 — Compute min/max/avg per state for each ingredient

# Merge price + premise to get state column
df = df_price.merge(df_premise[['premise_code', 'state']], on='premise_code', how='left')
df['state'] = df['state'].replace(STATE_NORMALISE).fillna('Unknown')

results = {}   # our_id → {state → {min, max, avg}}
national = {}  # our_id → {runcit_avg, runcit_min, runcit_max}

for our_id, cfg in ITEM_MAP.items():
    subset = df[df['item_code'].isin(cfg['codes'])].copy()
    if subset.empty:
        print(f'⚠️  No data for {our_id}')
        continue
    
    # Normalise to per-kg price
    subset = subset.copy()
    subset['price_per_kg'] = subset['price'] / cfg['unit_kg']
    
    # National stats
    national[our_id] = {
        'runcit_avg': round(subset['price_per_kg'].mean(), 2),
        'runcit_min': round(subset['price_per_kg'].min(), 2),
        'runcit_max': round(subset['price_per_kg'].max(), 2),
        'pasar_borong_est': round(subset['price_per_kg'].mean() * 0.75, 2),  # -25% wholesale est
        'observation_count': len(subset),
    }
    
    # Per-state stats
    state_grp = subset.groupby('state')['price_per_kg'].agg(['min', 'max', 'mean', 'count']).round(2)
    results[our_id] = state_grp.rename(columns={'mean': 'avg'}).to_dict('index')

# Print summary table
print(f'\nPriceCatcher Live Data — {YYYYMM}')
print(f'{"Ingredient":<22} {"Runcit Min":<14} {"Runcit Avg":<14} {"Runcit Max":<14} {"Est Borong":<12} {"n"}')
print('-' * 90)
for our_id, n in national.items():
    print(f'{our_id:<22} RM{n["runcit_min"]:<12.2f} RM{n["runcit_avg"]:<12.2f} RM{n["runcit_max"]:<12.2f} RM{n["pasar_borong_est"]:<10.2f} {n["observation_count"]}')

⚠️  No data for beras_basmati

PriceCatcher Live Data — 2026-04
Ingredient             Runcit Min     Runcit Avg     Runcit Max     Est Borong   n
------------------------------------------------------------------------------------------
ayam_standard          RM5.00         RM8.37         RM111.59       RM6.28       18686
daging_lembu           RM20.50        RM43.38        RM79.90        RM32.54      204
santan_segar           RM4.50         RM11.08        RM20.00        RM8.31       873
bawang_merah           RM0.60         RM7.18         RM22.00        RM5.39       14947
bawang_putih           RM0.50         RM9.10         RM20.00        RM6.83       24837
cili_kering            RM11.00        RM20.55        RM48.00        RM15.41      4939
lengkuas               RM0.89         RM10.37        RM24.00        RM7.78       21853
kunyit_hidup           RM0.07         RM9.82         RM79.00        RM7.37       22497
halia                  RM2.90         RM8.69         RM20.00        RM6

In [7]:
# Cell 5 — Preview per-state breakdown for a specific ingredient
# Change INGREDIENT_ID to inspect any ingredient

INGREDIENT_ID = 'ayam_standard'

if INGREDIENT_ID in results:
    print(f'\nPer-state breakdown: {INGREDIENT_ID}')
    print(f'{"State":<30} {"Min RM":<10} {"Avg RM":<10} {"Max RM":<10} {"Obs"}')
    print('-' * 65)
    for state, stats in sorted(results[INGREDIENT_ID].items()):
        print(f'{state:<30} {stats["min"]:<10.2f} {stats["avg"]:<10.2f} {stats["max"]:<10.2f} {int(stats["count"])}')


Per-state breakdown: ayam_standard
State                          Min RM     Avg RM     Max RM     Obs
-----------------------------------------------------------------
Johor                          5.00       7.82       11.30      2527
Kedah                          5.30       8.39       11.00      1361
Kelantan                       5.99       8.29       10.80      1646
Kuala Lumpur                   5.95       8.16       11.00      2126
Labuan                         12.50      12.50      12.50      22
Melaka                         6.20       8.24       10.30      526
Negeri Sembilan                5.99       8.19       11.50      764
Pahang                         5.00       8.26       12.60      1762
Perak                          5.95       8.24       12.00      2189
Perlis                         6.63       8.59       10.70      342
Pulau Pinang                   5.99       8.93       12.50      1429
Putrajaya                      5.80       8.10       11.00      345
Sabah   

In [8]:
# Cell 6 — WRITE to ingredients.json
# Review Cell 4 output before running!

import json
from pathlib import Path

KB_PATH = Path('../backend/app/knowledge_base/ingredients.json')

with open(KB_PATH, encoding='utf-8') as f:
    ingredients_data = json.load(f)

updated = []
skipped = []

for ing in ingredients_data['ingredients']:
    ing_id = ing['id']
    if ing_id not in national:
        skipped.append(ing_id)
        continue
    
    n = national[ing_id]
    ing['pasar_borong_price_myr'] = n['pasar_borong_est']  # wholesale estimate
    ing['retail_price_myr']       = n['runcit_avg']         # live national average
    ing['price_data_live'] = {
        'source': f'PriceCatcher {YYYYMM}',
        'runcit_national_avg': n['runcit_avg'],
        'runcit_national_min': n['runcit_min'],
        'runcit_national_max': n['runcit_max'],
        'pasar_borong_est': n['pasar_borong_est'],
        'observations': n['observation_count'],
        'by_state': {
            state: {'avg': round(stats['avg'], 2), 'min': round(stats['min'], 2)}
            for state, stats in results[ing_id].items()
            if state != 'Unknown'
        }
    }
    updated.append(ing_id)

ingredients_data['last_updated'] = str(date.today())
ingredients_data['source'] = f'PriceCatcher data.gov.my — {YYYYMM}'

with open(KB_PATH, 'w', encoding='utf-8') as f:
    json.dump(ingredients_data, f, ensure_ascii=False, indent=2)

print(f'✅ Updated {len(updated)} ingredients: {updated}')
print(f'   Skipped (no PriceCatcher data): {skipped}')
print(f'   File: {KB_PATH.resolve()}')

✅ Updated 10 ingredients: ['beras_wangi', 'daging_lembu', 'santan_segar', 'bawang_merah', 'bawang_putih', 'cili_kering', 'lengkuas', 'kunyit_hidup', 'minyak_masak', 'gula_pasir']
   Skipped (no PriceCatcher data): ['beras_basmati', 'beras_pulut', 'ayam_broiler', 'ayam_kampung', 'daging_lembu_import', 'daging_kambing', 'udang_harimau', 'udang_sederhana', 'santan_kotak_kara', 'serai', 'kerisik', 'gula_melaka']
   File: C:\Users\SYAQIRAH\Desktop\Dev-Master-Side-Project\projects\python\KenduriLuhh\backend\app\knowledge_base\ingredients.json
